# Atividade Formativa semana 5: Auditoria de Modelos

## Pergunta 1: Classificação (Breast Cancer)
A avaliação de sistemas de saúde não pode depender exclusivamente da Acurácia devido ao alto custo sistêmico de Falsos Negativos (pacientes doentes classificados como saudáveis). Abaixo, auditamos o modelo Random Forest utilizando a Matriz de Confusão (para mapear os erros reais), o F1-Score (média harmônica entre precisão e recall) e a métrica AUC-ROC (capacidade do motor de separar as classes).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

# 1. Ingestão e Preparação (Pipeline da Semana 3)
df_cancer = pd.read_csv('wisconsin.csv')
df_cancer = df_cancer.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

le = LabelEncoder()
df_cancer['diagnosis'] = le.fit_transform(df_cancer['diagnosis'])

X_c = df_cancer.drop(columns=['diagnosis'])
y_c = df_cancer['diagnosis']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

# 2. Treinamento do Random Forest
modelo_rf_cancer = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf_cancer.fit(X_train_c_scaled, y_train_c)

# 3. Auditoria (Previsões e Probabilidades)
previsoes_rf = modelo_rf_cancer.predict(X_test_c_scaled)
probs_rf = modelo_rf_cancer.predict_proba(X_test_c_scaled)[:, 1]

print("--- AUDITORIA: RANDOM FOREST (CÂNCER) ---")
print("Matriz de Confusão:\n", confusion_matrix(y_test_c, previsoes_rf))
print(f"F1-Score: {f1_score(y_test_c, previsoes_rf):.4f}")
print(f"AUC (Área Sob a Curva ROC): {roc_auc_score(y_test_c, probs_rf):.4f}")

--- AUDITORIA: RANDOM FOREST (CÂNCER) ---
Matriz de Confusão:
 [[70  1]
 [ 3 40]]
F1-Score: 0.9524
AUC (Área Sob a Curva ROC): 0.9953


## Pergunta 2: Regressão (Expectativa de Vida)
Para prever variáveis contínuas que afetam políticas públicas, o modelo deve ser avaliado pela sua margem de erro estrutural. O MAE nos dá o erro médio em anos, enquanto o RMSE penaliza severamente os erros grandes (outliers). O R² confirma qual a porcentagem da realidade que o modelo consegue explicar com os dados fornecidos.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Ingestão e Saneamento
df_vida = pd.read_csv('expectancy.csv')
df_vida.columns = df_vida.columns.str.strip()
df_vida = df_vida.dropna(subset=['Life expectancy'])

X_v = df_vida.drop(columns=['Life expectancy', 'Country'])
y_v = df_vida['Life expectancy']

cols_num = X_v.select_dtypes(include=['int64', 'float64']).columns
cols_cat = X_v.select_dtypes(include=['object']).columns

# 2. Arquitetura do Pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), cols_num),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cols_cat)
])

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_v, y_v, test_size=0.2, random_state=42)

pipeline_reg = Pipeline(steps=[('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42))])
pipeline_reg.fit(X_train_v, y_train_v)

# 3. Auditoria
previsoes_vida = pipeline_reg.predict(X_test_v)

print("--- AUDITORIA: RANDOM FOREST (EXPECTATIVA DE VIDA) ---")
print(f"MAE (Erro Médio Absoluto): {mean_absolute_error(y_test_v, previsoes_vida):.4f} anos")
print(f"MSE (Erro Quadrático): {mean_squared_error(y_test_v, previsoes_vida):.4f}")
# Hack de infraestrutura: extração de raiz manual para evitar crash de versão
rmse_seguro = mean_squared_error(y_test_v, previsoes_vida) ** 0.5
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse_seguro:.4f} anos")
print(f"R² (Capacidade Explicativa): {r2_score(y_test_v, previsoes_vida):.2%}")

/tmp/ipykernel_15464/2302766060.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = X_v.select_dtypes(include=['object']).columns


--- AUDITORIA: RANDOM FOREST (EXPECTATIVA DE VIDA) ---
MAE (Erro Médio Absoluto): 1.0476 anos
MSE (Erro Quadrático): 2.7857
RMSE (Raiz do Erro Quadrático Médio): 1.6690 anos
R² (Capacidade Explicativa): 96.78%


## Pergunta 3: Comparativo de Performance (Random Forest vs XGBoost)
Utilizando o dataset de classificação (Breast Cancer), submetemos os mesmos dados padronizados a um algoritmo baseado em Gradient Boosting (XGBoost) para comparar a resiliência contra o Random Forest.

In [3]:
import xgboost as xgb

# 1. Instanciamento e Treino do XGBoost (Aproveitando os dados particionados da Célula 2)
modelo_xgb_cancer = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=4, 
    random_state=42,
    eval_metric='logloss'
)
modelo_xgb_cancer.fit(X_train_c_scaled, y_train_c)

# 2. Coleta de Métricas
previsoes_xgb = modelo_xgb_cancer.predict(X_test_c_scaled)
probs_xgb = modelo_xgb_cancer.predict_proba(X_test_c_scaled)[:, 1]

f1_rf = f1_score(y_test_c, previsoes_rf)
auc_rf = roc_auc_score(y_test_c, probs_rf)
f1_xgb = f1_score(y_test_c, previsoes_xgb)
auc_xgb = roc_auc_score(y_test_c, probs_xgb)

print("--- AUDITORIA: XGBOOST (CÂNCER) ---")
print("Matriz de Confusão:\n", confusion_matrix(y_test_c, previsoes_xgb))
print(f"F1-Score XGBoost: {f1_xgb:.4f} | F1-Score RF: {f1_rf:.4f}")
print(f"AUC XGBoost:      {auc_xgb:.4f} | AUC RF:      {auc_rf:.4f}\n")

# 3. Laudo Analítico de Comparação
if auc_xgb > auc_rf and f1_xgb > f1_rf:
    vencedor = "XGBoost"
elif auc_rf > auc_xgb and f1_rf > f1_xgb:
    vencedor = "Random Forest"
else:
    vencedor = "Empate Técnico (Depende do limiar de operação corporativa)"

print("--- DIAGNÓSTICO FINAL ---")
print(f"O melhor modelo ao comparar todas as métricas simultaneamente foi: {vencedor}.")
print("Analisando individualmente, o algoritmo que maximiza o AUC e o F1-Score demonstra maior capacidade matemática de minimizar os Falsos Positivos e Falsos Negativos sistêmicos.")

--- AUDITORIA: XGBOOST (CÂNCER) ---
Matriz de Confusão:
 [[69  2]
 [ 3 40]]
F1-Score XGBoost: 0.9412 | F1-Score RF: 0.9524
AUC XGBoost:      0.9938 | AUC RF:      0.9953

--- DIAGNÓSTICO FINAL ---
O melhor modelo ao comparar todas as métricas simultaneamente foi: Random Forest.
Analisando individualmente, o algoritmo que maximiza o AUC e o F1-Score demonstra maior capacidade matemática de minimizar os Falsos Positivos e Falsos Negativos sistêmicos.
